In [2]:
from functools import partial
import jax
import jax.numpy as jnp
import numpy as np
import queue
import threading


# ---------------------------------------------
# 1) JIT-compiled "replace" and "sample" ops
# ---------------------------------------------
@jax.jit
def replace_cache_jit(loader_state, new_cache):
    """
    Replace the old cache entirely with 'new_cache'.
    Returns an updated loader_state dict.
    """
    return {
        "cache": new_cache,
        "rng":   loader_state["rng"],
    }

@partial(jax.jit, static_argnums=(1,))
def sample_cache_jit(loader_state, batch_size):
    """
    Sample a random minibatch of 'batch_size' from loader_state['cache'].
    Returns (new_loader_state, batch).
    """
    rng = loader_state["rng"]
    rng, subkey = jax.random.split(rng)

    cache = loader_state["cache"]
    cache_size = cache.shape[0]
    idxs = jax.random.randint(subkey, (batch_size,), 0, cache_size)

    batch = cache[idxs, ...]  # fancy indexing
    new_loader_state = {
        "cache": cache,
        "rng":   rng,
    }
    return new_loader_state, batch


# ---------------------------------------------
# 2) SyntheticDataLoader (Double-Buffer style)
# ---------------------------------------------
class SyntheticDataLoader:
    """
    A data loader that:
      - Maintains a single "current" cache (size = cache_size).
      - A background thread produces a *new* full cache in the queue (one big batch).
      - As soon as that big batch is ready, we *swap* out the old cache and replace it.
      - We sample randomly from the current cache for training.

    No partial/rolling updates -- each time we replace, it's a full new dataset.
    """

    def __init__(
        self,
        simulator,
        cache_size=1024,
        data_shape=(28, 28),
        max_queue_size=2,
        store_on_device=True,
        seed=42,
    ):
        """
        :param simulator: a function with signature:
            simulator(rng_key, batch_size, data_shape) -> (new_rng_key, data)
          Must return a JAX or NumPy array of shape (batch_size, *data_shape).
          We'll call it with batch_size = cache_size each time.
        :param cache_size: number of samples to store in the cache (and produce at once).
        :param data_shape: shape of a single data sample.
        :param max_queue_size: queue capacity for "full-caches" that are produced.
        :param store_on_device: if True, store the cache as a JAX array on GPU/TPU
                                (depending on your default JAX device).
        :param seed: base random seed for both simulator RNG and sampling RNG.
        """
        self.simulator     = simulator
        self.cache_size    = cache_size
        self.data_shape    = data_shape
        self.store_on_device = store_on_device

        # Create an initial empty cache on device:
        cache = jnp.zeros((cache_size,) + data_shape, dtype=jnp.float32)

        # We'll store this in a "loader_state" dict
        self.loader_state = {
            "cache": cache,
            "rng":   jax.random.PRNGKey(seed),  # for sampling from the cache
        }

        # Separate RNG for the simulator in the background thread
        self.simulator_rng = jax.random.PRNGKey(seed + 10_000)

        # A queue where each entry is a *full new cache* of shape (cache_size, ...).
        # We'll push from the background thread, pull in the main thread.
        self.queue = queue.Queue(maxsize=max_queue_size)

        # Create & start background thread
        self._stop_event = threading.Event()
        self._thread = threading.Thread(
            target=self._producer_loop,
            daemon=True
        )
        self._thread.start()

    def _producer_loop(self):
        """Background thread that produces a *full new cache* (cache_size) each iteration."""
        while not self._stop_event.is_set():
            # Produce a single batch of size = cache_size
            self.simulator_rng, new_data = self.simulator(
                self.simulator_rng,
                self.cache_size,
                self.data_shape
            )
            # Convert to NumPy on CPU if it's a jax.Array
            new_data = np.array(new_data, copy=False)

            # Put it in the queue (blocks if full)
            self.queue.put(new_data)

    def init_cache(self):
        """
        Optionally fill the cache immediately (blocking) before training
        so you start with a "warm" cache.
        """
        self.simulator_rng, big_batch = self.simulator(
            self.simulator_rng,
            self.cache_size,
            self.data_shape
        )
        big_batch_jax = jnp.array(big_batch)  # ensure a jax Array
        self.loader_state = replace_cache_jit(self.loader_state, big_batch_jax)

    def maybe_replace_cache(self):
        """
        If there's a *complete new cache* in the queue, replace the old one.
        Otherwise, do nothing.
        """
        if not self.queue.empty():
            # Pull the new data
            new_batch_np = self.queue.get()
            # Convert to jax array
            new_batch_jax = jnp.array(new_batch_np)
            # Swap out old cache for new
            self.loader_state = replace_cache_jit(self.loader_state, new_batch_jax)

    def get_batch_from_cache(self, batch_size):
        """
        1) Possibly replace the entire cache if the background thread has one ready.
        2) Sample from the current cache (JIT-compiled).
        """
        # Step 1: see if there's a new cache in the queue
        self.maybe_replace_cache()

        # Step 2: sample
        self.loader_state, batch = sample_cache_jit(self.loader_state, batch_size)
        return batch

    def stop(self):
        """Stop the background producer."""
        self._stop_event.set()
        self._thread.join()


# ---------------------------------------------
# 3) Example simulator function
# ---------------------------------------------
@partial(jax.jit, static_argnums=(1,2))
def my_simulator(rng_key, batch_size, data_shape):
    """
    Just returns random normal data. You can replace
    with your own synthetic generation code.
    """

    data = 0
    for _ in range(100):
        rng_key, subkey = jax.random.split(rng_key)
        data +=jax.random.normal(subkey, shape=(batch_size,) + data_shape)
    return rng_key, data


# ---------------------------------------------
# 4) Example usage
# ---------------------------------------------
def main():
    # Create a loader that uses the "my_simulator" function,
    # produces entire 10_000-sample caches, and swaps them out in full.
    loader = SyntheticDataLoader(
        simulator=my_simulator,
        cache_size=10_000,
        data_shape=(28, 28),
        store_on_device=True,
        seed=42
    )

    # Optionally fill the cache once right away
    loader.init_cache()

    # Example trivial train loop
    @jax.jit
    def train_step(params, x):
        # Just sum-of-squares loss for demonstration
        loss = jnp.mean(x * x)
        return params, loss

    params = jnp.array(0.0)  # dummy
    for step in range(5):
        # Get a mini-batch of size 32 from the current (possibly just swapped) cache
        batch = loader.get_batch_from_cache(batch_size=32)
        params, loss = train_step(params, batch)
        print(f"Step {step}, loss = {loss}")

    loader.stop()


In [3]:
loader = SyntheticDataLoader(
        simulator=my_simulator,
        cache_size=10_000,
        data_shape=(10,),
        store_on_device=True,
        seed=42
    )
#loader.init_cache()

In [4]:
def build_producer_loop(queue, simulator, event, rng, batch_size=32, device="cpu", device_idx=0):
    """Background thread that produces a *full new cache* (cache_size) each iteration."""
    device = jax.devices(device)[device_idx]
    rng = jax.device_put(rng, device)

    # Batch simulator
    @partial(jax.jit, device=device)
    def batch_simulator(rng):
        rngs = jax.random.split(rng, batch_size)
        return jax.vmap(simulator)(rngs)
    def _producer_loop():
        with jax.default_device(device):
            key = rng
            while not event.is_set():
                # Produce a single batch of size = cache_size
                key, rng_sub = jax.random.split(key)
                data = batch_simulator(rng_sub)
                # Convert to NumPy on CPU if it's a jax.Array
                data = np.array(data, copy=False)

                # Put it in the queue (blocks if full)
                queue.put(data)
    return _producer_loop

def build_data_stream(queue, event):
    while not event.is_set():
        if not queue.empty():
            yield queue.get()

# Create & start background thread
def simulator(rng):
    print("simulating")
    data = jax.random.normal(rng, shape=(10,))
    data = jax.lax.scan(lambda x, _: (x + jax.random.normal(rng, shape=(10,)), 0), data, jnp.arange(100))[0]
    return data

q = queue.Queue(maxsize=1)
event = threading.Event()
thread = threading.Thread(
            target=build_producer_loop(q, simulator, event, jax.random.key(0)),
            daemon=True
        )

thread.start()

data = build_data_stream(q, event)

In [5]:
# Create & start background thread
def simulator(rng):
    print("simulating")
    data = jax.random.normal(rng, shape=(10,))
    data = jax.lax.scan(lambda x, _: (x + jax.random.normal(rng, shape=(10,)), 0), data, jnp.arange(10))[0]
    return data

q = queue.Queue(maxsize=1)
event = threading.Event()
thread = threading.Thread(
            target=build_producer_loop(q, simulator, event, jax.random.key(0)),
            daemon=True
        )

thread.start()

data = build_data_stream(q, event)

simulating


simulating


In [6]:
%%time
simulator(jax.random.key(0))

simulating
CPU times: user 1.78 s, sys: 238 ms, total: 2.02 s
Wall time: 548 ms


Array([ 17.849066  ,  22.277912  ,  -4.7695384 ,  -0.86479074,
         1.937     , -10.692981  ,  -5.448287  ,   5.4381647 ,
         7.307843  , -10.451798  ], dtype=float32)

In [7]:
%%time
next(data)

CPU times: user 40 μs, sys: 0 ns, total: 40 μs
Wall time: 44.1 μs


array([[  2.1879816 ,  -8.471947  , -13.846062  ,  -4.5164375 ,
        -11.062246  ,   3.152081  ,  10.940337  ,   5.4813623 ,
         -8.178061  ,  12.526471  ],
       [-14.691874  ,  -3.878193  ,   3.4067445 , -11.0104065 ,
         -0.85247344, -10.012777  ,   2.9342241 ,  19.947216  ,
         -2.674141  , -10.460779  ],
       [  7.4761972 ,   8.8756895 , -10.1712475 ,  -9.353119  ,
        -11.135558  ,  14.218188  ,   1.5903718 ,  -5.527264  ,
         -0.32632577,  -4.7452197 ],
       [ 18.269638  ,   2.8110676 ,   0.31847158,  15.17613   ,
         28.581137  ,  -0.32721698,   1.8988283 , -11.035594  ,
        -17.984303  ,  12.990965  ],
       [  7.694369  , -11.276981  ,   5.3722463 ,  -2.0457497 ,
          1.0420603 ,  -7.7963953 , -24.605608  ,  -0.0551741 ,
         13.351743  ,   7.775624  ],
       [  8.317437  ,  -6.602868  ,  17.661213  ,  20.65298   ,
          0.21527013,  20.910862  , -12.549488  ,  12.381109  ,
          0.9157227 ,   7.2916894 ],
       [-1

In [8]:
%%timeit
np.array([q.get() for _ in range(5)])

KeyboardInterrupt: 

In [8]:
q.get()

array([[  1.1628466 , -11.17992   ,   2.7457328 ,  -2.4355814 ,
        -12.180299  ,  -3.3037033 ,   6.5850253 ,  -9.392833  ,
         -0.68319625, -19.448322  ],
       [ 17.17122   ,   9.432581  ,  24.326105  , -15.235045  ,
         -8.222178  ,   7.3275146 ,  24.204987  ,  -3.9701033 ,
          2.753543  ,   6.973816  ],
       [  0.74308026,  -5.43458   , -12.740756  ,   0.7987103 ,
         -8.795571  ,  -0.8594034 , -13.629662  ,  -1.9215808 ,
          0.591927  , -10.608782  ],
       [-11.412388  ,  10.193603  ,  15.488881  ,  12.649649  ,
          9.685862  ,   6.067742  ,   4.1106915 ,  11.582144  ,
        -16.906933  ,  -2.8543227 ],
       [-14.639419  ,   1.0671494 ,   6.4992757 ,  -0.2759476 ,
        -34.807358  ,  -4.7547994 ,  -0.63283896,  15.444214  ,
         14.396922  ,  12.8344345 ],
       [-11.376513  ,   0.95188093, -20.410835  ,  10.066595  ,
          2.2853882 ,  -4.822891  ,  24.771637  ,  -2.583124  ,
         -9.557952  ,  -6.5773907 ],
       [ -

In [9]:
%%timeit
loader.get_batch_from_cache(batch_size=1024).block_until_ready()

1.48 ms ± 88.8 μs per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [10]:
my_simulator

<PjitFunction of <function my_simulator at 0x7f6db4a7e480>>

In [11]:
loader.get_batch_from_cache(batch_size=32)

Array([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 

In [ ]:
import threading
import queue
from functools import partial

import jax
import jax.numpy as jnp
import numpy as np


class StreaDataLoader:
    """
    A simple JAX DataLoader-like class that:
      1) Runs a background thread to produce CPU data batches.
      2) Prefetches each batch to the GPU (or specified device)
         just-in-time while the GPU is busy training on the previous batch.
    """

    def __init__(
        self,
        simulator_fn,
        rng=None,
        batch_size=32,
        max_queue_size=2,
        data_device="gpu",
        simulation_device="cpu",
        device_idx=0,
        daemon=True
    ):
        """
        Args:
            simulator_fn: A function simulator(rng) -> batch_of_data
                          or a jitted function that produces a single item.
                          We'll vmap/jit over it in a background thread.
            rng:          JAX PRNGKey to seed the simulator.
            batch_size:   Number of items per batch.
            max_queue_size: Max items the queue can hold.
            device:       "cpu" or "gpu" (passed to jax.devices()).
            device_idx:   Index of the device (0 for first GPU, etc.).
            daemon:       Whether the producer thread is daemonized.
        """
        self.simulator_fn = simulator_fn
        if rng is None:
            rng = jax.random.PRNGKey(0)
        self.rng = rng
        self.batch_size = batch_size
        self.device = jax.devices(data_device)[device_idx]
        self.simulation_device = jax.devices(simulation_device)[device_idx]

        # For the background thread
        self.event = threading.Event()
        self.queue = queue.Queue(maxsize=max_queue_size)

        # Build the producer thread, which will feed CPU data
        self.producer_thread = threading.Thread(
            target=self._producer_loop,
            daemon=daemon
        )
        # Start the thread
        self.producer_thread.start()

    def _producer_loop(self):
        """
        The background loop that repeatedly:
          - Splits the RNG
          - Produces a batch on the device
          - Transfers it back to CPU (numpy array)
          - Puts it into the CPU queue
        """
        # We'll jit+vmap the user simulator to produce batch_size items at once
        @partial(jax.jit, device=self.simulation_device)
        def batch_simulator(rng_key):
            rngs = jax.random.split(rng_key, self.batch_size)
            return jax.vmap(self.simulator_fn)(rngs)

        key = jax.device_put(self.rng, self.simulation_device)
        while not self.event.is_set():
            # Generate a batch
            key, rng_sub = jax.random.split(key)
            data = batch_simulator(rng_sub)

            # Convert jax.Array -> NumPy (CPU) for queuing
            # so we don't hold a big device buffer in the queue
            data_cpu = np.array(data, copy=False)

            # Blocks if queue is full
            self.queue.put(data_cpu)

    def _cpu_data_stream(self):
        """
        A generator that yields batches from the CPU queue.
        Ends if the event is set.
        """
        while not self.event.is_set():
            # Blocks if queue is empty
            batch_cpu = self.queue.get()
            yield batch_cpu



    def _prefetch_data_stream(self, cpu_stream):
        """
        Generator that:
          - Pulls CPU batches from 'cpu_stream'
          - Asynchronously copies them to `self.device` using `_transfer_to_device()`
          - Yields them (already on the device)
          - Prefetches the next batch in parallel
        """
        stream_iter = iter(cpu_stream)

        # Try to grab first batch
        try:
            next_batch_on_gpu = self._transfer_to_device(next(stream_iter))
        except StopIteration:
            return  # If no data, we're done

        while True:
            current_batch = next_batch_on_gpu  # The batch to yield
            # Prefetch the next batch
            try:
                cpu_batch = next(stream_iter)
                next_batch_on_gpu = self._transfer_to_device(cpu_batch)
            except StopIteration:
                next_batch_on_gpu = None

            yield current_batch

            if next_batch_on_gpu is None:
                break


    def __iter__(self):
        """
        Returns a generator that yields GPU batches, prefetching them in parallel.
        """
        cpu_stream = self._cpu_data_stream()
        # `yield from` the prefetch generator
        yield from self._prefetch_data_stream(cpu_stream)

    def close(self):
        """
        Signal the producer thread to exit and wait for it to join.
        """
        self.event.set()
        if self.producer_thread.is_alive():
            self.producer_thread.join()

    def __del__(self):
        """
        Make sure the background thread is stopped if the loader is garbage-collected.
        """
        self.close()


# -----------------------------------------------------------------------------
# Example usage
# -----------------------------------------------------------------------------

def example_simulator(rng):
    """
    Example user-defined simulator function that returns a single data item
    based on rng. We'll vmap over it to get a batch.
    """
    # For demonstration, just some random normal
    data = jax.random.normal(rng, shape=(10,))
    # Let's do a scan to add a bit more overhead
    data = jax.lax.scan(
        lambda x, _: (x + jax.random.normal(rng, shape=(10,)), None),
        data,
        jnp.arange(1000)
    )[0]
    return data

loader = StreaDataLoader(
    simulator_fn=example_simulator,
    rng=jax.random.PRNGKey(42),
    batch_size=32,
    max_queue_size=100000,
    simulation_device="cpu",
    device_idx=0,
    daemon=True
)


In [13]:
import threading
import queue
from functools import partial

import jax
import jax.numpy as jnp
import numpy as np


class StreaDataLoader:
    """
    A JAX DataLoader-like class with two modes of "prefetch" to the data device:

    1) Simple device_put per batch (default).
    2) Ring buffer / in-place updates (use_ring_buffer=True).

    In either mode:
      - A background thread produces CPU data in a queue.
      - We consume from that queue, copy data to `data_device`, and yield.
      - The next batch is prefetched while the current one is being processed.
    """

    def __init__(
        self,
        simulator_fn,
        rng=None,
        batch_size=32,
        max_queue_size=2,
        data_device="gpu",
        simulation_device="cpu",
        device_idx=0,
        daemon=True,
        use_ring_buffer=False,
        ring_size=2,
    ):
        """
        Args:
            simulator_fn: A function (rng) -> single item. We'll vmap over `batch_size`
                          in a background thread to produce batches.
            rng:          JAX PRNGKey to seed the simulator (default PRNGKey(0)).
            batch_size:   Number of items per batch.
            max_queue_size: Maximum items the queue can hold at once.
            data_device:  The device where final data ends up (e.g. "gpu").
            simulation_device: Device used to run the simulation (often "cpu").
            device_idx:   Index of the device (0 for first GPU, etc.).
            daemon:       Whether the producer thread is daemonized.
            use_ring_buffer:  If True, use pre-allocated device buffers + in-place updates.
            ring_size:        Number of device buffers to allocate if using a ring.
        """
        self.simulator_fn = simulator_fn
        self.rng = rng or jax.random.PRNGKey(0)
        self.batch_size = batch_size

        # Device for final training data (e.g. GPU)
        self.data_device = jax.devices(data_device)[device_idx]
        # Device to generate batches (often CPU, but can be GPU)
        self.simulation_device = jax.devices(simulation_device)[device_idx]

        self.event = threading.Event()
        self.queue = queue.Queue(maxsize=max_queue_size)

        # Ring buffer options
        self.use_ring_buffer = use_ring_buffer
        self.ring_size = ring_size
        self._ring_initialized = False
        self.gpu_ring_buffers = None  # Will hold device arrays if using ring

        # Producer thread that generates CPU batches
        self.producer_thread = threading.Thread(
            target=self._producer_loop,
            daemon=daemon
        )
        self.producer_thread.start()

    def _producer_loop(self):
        """
        Background loop:
          - Splits RNG
          - Produces a batch on self.simulation_device (jit+vmap)
          - Converts to CPU (NumPy)
          - Puts into self.queue
        """
        @partial(jax.jit, device=self.simulation_device)
        def batch_simulator(rng_key):
            rngs = jax.random.split(rng_key, self.batch_size)
            return jax.vmap(self.simulator_fn)(rngs)

        key = jax.device_put(self.rng, self.simulation_device)

        while not self.event.is_set():
            key, rng_sub = jax.random.split(key)
            data = batch_simulator(rng_sub)

            # Convert from jax.Array -> CPU NumPy
            data_cpu = np.array(data, copy=False)

            # Block if queue is full
            self.queue.put(data_cpu)

    def _cpu_data_stream(self):
        """
        Yields CPU (NumPy) batches from self.queue.
        Ends if the event is set.
        """
        while not self.event.is_set():
            batch_cpu = self.queue.get()
            yield batch_cpu

    # -----------------------------
    # Mode A: Normal device_put
    # -----------------------------
    def _transfer_to_device(self, batch_cpu):
        """Simple CPU -> device copy."""
        return jax.device_put(batch_cpu, self.data_device)

    def _prefetch_data_stream_device_put(self, cpu_stream):
        """
        Generator that:
          - Grabs CPU batches from cpu_stream
          - device_put to self.data_device
          - Prefetches next while current is used
        """
        stream_iter = iter(cpu_stream)

        try:
            next_on_dev = self._transfer_to_device(next(stream_iter))
        except StopIteration:
            return

        while True:
            current_batch = next_on_dev
            try:
                cpu_batch = next(stream_iter)
                next_on_dev = self._transfer_to_device(cpu_batch)
            except StopIteration:
                next_on_dev = None

            yield current_batch

            if next_on_dev is None:
                break

    # -----------------------------
    # Mode B: Ring buffer approach
    # -----------------------------
    @staticmethod
    @partial(jax.jit, donate_argnums=(0,))
    def _copy_inplace(dst, new_data):
        """
        Copy `new_data` (CPU or device array) into `dst` in-place.
        `dst.at[...].set(...)` must match shape exactly.

        Using donate_argnums=(0,) so XLA can reuse memory. Typically helpful
        if we no longer need the old contents of `dst`.
        """
        return dst.at[:].set(new_data)

    def _init_ring_buffers(self, batch_cpu):
        """
        Allocate ring_size device buffers with the same shape/dtype as `batch_cpu`.
        """
        self._ring_initialized = True
        shape = batch_cpu.shape
        dtype = batch_cpu.dtype

        self.gpu_ring_buffers = []
        for _ in range(self.ring_size):
            # Pre-allocate zero arrays on data_device
            arr = jax.device_put(jnp.zeros(shape, dtype), self.data_device)
            self.gpu_ring_buffers.append(arr)

    def _prefetch_data_stream_ring(self, cpu_stream):
        """
        Generator that:
          - Pulls CPU batches
          - Copies them in-place to ring buffers on the device
          - Yields them, prefetching next in a ring fashion
        """
        stream_iter = iter(cpu_stream)
        ring_idx = 0

        # Initialize ring on the first batch
        try:
            first_batch_cpu = next(stream_iter)
        except StopIteration:
            return

        self._init_ring_buffers(first_batch_cpu)

        # Copy first batch
        buf = self._copy_inplace(self.gpu_ring_buffers[ring_idx], first_batch_cpu)
        self.gpu_ring_buffers[ring_idx] = buf
        yield buf

        ring_idx = (ring_idx + 1) % self.ring_size

        # Subsequent batches
        for batch_cpu in stream_iter:
            buf = self._copy_inplace(self.gpu_ring_buffers[ring_idx], batch_cpu)
            self.gpu_ring_buffers[ring_idx] = buf
            yield buf
            ring_idx = (ring_idx + 1) % self.ring_size

    # -----------------------------
    # Iterator
    # -----------------------------
    def __iter__(self):
        """
        Returns a generator that yields device-resident batches,
        either via normal device_put or ring-buffer in-place updates.
        """
        cpu_stream = self._cpu_data_stream()

        if self.use_ring_buffer:
            yield from self._prefetch_data_stream_ring(cpu_stream)
        else:
            yield from self._prefetch_data_stream_device_put(cpu_stream)

    def close(self):
        """
        Signal the producer thread to exit and wait for it to join.
        """
        self.event.set()
        if self.producer_thread.is_alive():
            self.producer_thread.join()

    def __del__(self):
        self.close()


In [14]:
import threading
import queue
from functools import partial

import jax
import jax.numpy as jnp
import numpy as np


class StreaDataLoader:
    """
    A JAX DataLoader-like class that:
      - Produces CPU data in a background thread (simulation_device).
      - Prefetches each batch to data_device (GPU, etc.).
      - Optionally uses in-place updates with ring buffers:
          use_inplace_update = True -> Allocate ring buffers, .at[..].set(...)
          use_inplace_update = False -> device_put each batch, no ring buffers.
    """

    def __init__(
        self,
        simulator_fn,
        rng=None,
        batch_size=32,
        max_queue_size=2,
        data_device="gpu",
        simulation_device="cpu",
        device_idx=0,
        daemon=True,
        use_inplace_update=False,
        ring_size=2,
    ):
        """
        Args:
            simulator_fn: A function (rng) -> single item (we vmap over batch_size).
            rng:          Optional JAX PRNGKey (default jax.random.PRNGKey(0)).
            batch_size:   Number of items per batch.
            max_queue_size: Max items in CPU queue.
            data_device:  "cpu", "gpu", etc. for final data location.
            simulation_device: "cpu"/"gpu" for generating data.
            device_idx:   Device index (0 for first GPU, etc.).
            daemon:       Whether the producer thread is daemonized.
            use_inplace_update: If True, allocate ring buffers on first batch and do in-place.
                                If False, no ring buffer is allocated; just device_put each batch.
            ring_size:    Number of buffers if using in-place. (Ignored if use_inplace_update=False)
        """
        self.simulator_fn = simulator_fn
        self.rng = rng or jax.random.PRNGKey(0)
        self.batch_size = batch_size

        self.data_device = jax.devices(data_device)[device_idx]
        self.simulation_device = jax.devices(simulation_device)[device_idx]

        self.use_inplace_update = use_inplace_update
        self.ring_size = ring_size

        self.event = threading.Event()
        self.queue = queue.Queue(maxsize=max_queue_size)

        # For ring-buffer usage, we create them after first batch
        self._ring_initialized = False
        self.gpu_ring_buffers = []

        # Start background producer
        self.producer_thread = threading.Thread(
            target=self._producer_loop,
            daemon=daemon
        )
        self.producer_thread.start()

    def _producer_loop(self):
        """
        Background loop:
          - Splits RNG
          - Produces a batch on simulation_device
          - Converts to CPU (NumPy)
          - Puts it into self.queue
        """
        @partial(jax.jit, device=self.simulation_device)
        def batch_simulator(rng_key):
            rngs = jax.random.split(rng_key, self.batch_size)
            return jax.vmap(self.simulator_fn)(rngs)

        key = jax.device_put(self.rng, self.simulation_device)

        while not self.event.is_set():
            key, rng_sub = jax.random.split(key)
            data = batch_simulator(rng_sub)
            data_cpu = np.array(data, copy=False)
            self.queue.put(data_cpu)  # block if full

    def _cpu_data_stream(self):
        """
        Yields CPU (NumPy) batches from self.queue. Terminates when event is set.
        """
        while not self.event.is_set():
            batch_cpu = self.queue.get()
            yield batch_cpu

    # --------------------------------------------------------------------------
    # Method A: No in-place updates, no ring buffer
    # --------------------------------------------------------------------------
    def _prefetch_data_stream_device_put(self, cpu_stream):
        """
        Generator that simply calls device_put for each incoming CPU batch
        and yields that device array. Prefetch is done one batch ahead.
        """
        stream_iter = iter(cpu_stream)

        try:
            next_batch_dev = jax.device_put(next(stream_iter), self.data_device)
        except StopIteration:
            return

        while True:
            current_batch = next_batch_dev
            try:
                cpu_batch = next(stream_iter)
                next_batch_dev = jax.device_put(cpu_batch, self.data_device)
            except StopIteration:
                next_batch_dev = None

            yield current_batch

            if next_batch_dev is None:
                break

    # --------------------------------------------------------------------------
    # Method B: In-place updates with ring buffer
    # --------------------------------------------------------------------------
    @staticmethod
    @partial(jax.jit, donate_argnums=(0,))
    def _copy_inplace(dst, src):
        """
        In-place update: copy src -> dst with .at[..].set(...)
        donate_argnums=(0,) means we don't need old dst contents anymore.
        """
        return dst.at[:].set(src)

    def _init_ring_buffers(self, first_batch_cpu):
        """
        Allocates ring_size device arrays to match shape/dtype of the first batch.
        """
        shape = first_batch_cpu.shape
        dtype = first_batch_cpu.dtype

        self.gpu_ring_buffers = []
        for _ in range(self.ring_size):
            # create a zero array on data_device
            arr = jax.device_put(jnp.zeros(shape, dtype), self.data_device)
            self.gpu_ring_buffers.append(arr)

        self._ring_initialized = True

    def _prefetch_data_stream_inplace(self, cpu_stream):
        """
        Generator that:
          - Allocates ring buffers on first batch (if not done)
          - Copies CPU data into ring buffers in-place
          - Yields them, prefetching the next
        """
        stream_iter = iter(cpu_stream)
        ring_idx = 0

        # Get the first batch
        try:
            first_batch_cpu = next(stream_iter)
        except StopIteration:
            return

        # Init ring buffer
        self._init_ring_buffers(first_batch_cpu)

        # Copy first batch in-place
        buf = self._copy_inplace(self.gpu_ring_buffers[ring_idx], first_batch_cpu)
        self.gpu_ring_buffers[ring_idx] = buf
        yield buf
        ring_idx = (ring_idx + 1) % self.ring_size

        # Subsequent batches
        for batch_cpu in stream_iter:
            buf = self._copy_inplace(self.gpu_ring_buffers[ring_idx], batch_cpu)
            self.gpu_ring_buffers[ring_idx] = buf
            yield buf
            ring_idx = (ring_idx + 1) % self.ring_size

    # --------------------------------------------------------------------------
    # __iter__ that picks the correct method
    # --------------------------------------------------------------------------
    def __iter__(self):
        """
        Yields device-resident batches, either by:
          - direct device_put (no ring buffer)
          - or ring buffer in-place updates
        """
        cpu_stream = self._cpu_data_stream()

        if self.use_inplace_update:
            yield from self._prefetch_data_stream_inplace(cpu_stream)
        else:
            yield from self._prefetch_data_stream_device_put(cpu_stream)

    def close(self):
        """
        Signal the producer thread to exit and wait for it to join.
        """
        self.event.set()
        if self.producer_thread.is_alive():
            self.producer_thread.join()

    def __del__(self):
        self.close()


In [23]:
def example_simulator(rng):
    """
    Example user-defined simulator function that returns a single data item
    based on rng. We'll vmap over it to get a batch.
    """
    # For demonstration, just some random normal
    data = jax.random.normal(rng, shape=(10,))
    # Let's do a scan to add a bit more overhead
    data = jax.lax.scan(
        lambda x, _: (x + jax.random.normal(rng, shape=(10,)), None),
        data,
        jnp.arange(10)
    )[0]
    return data

In [24]:
import threading
import queue
import time
from functools import partial
from contextlib import contextmanager

import jax
import jax.numpy as jnp
import numpy as np


class StreamDataLoader:
    """
    A simple JAX DataLoader-like class that:
      1) Runs background threads to produce CPU data batches.
      2) Prefetches each batch to the GPU (or specified device)
         just-in-time while the GPU is busy training on the previous batch.
    """

    def __init__(
        self,
        simulator_fn,
        rng=None,
        batch_size=32,
        max_queue_size=2,
        data_device="gpu",
        simulation_device="cpu",
        device_idx=0,
        daemon=True,
        num_producers=1,
        queue_timeout=None,  # New parameter for queue timeout
    ):
        """
        Args:
            simulator_fn: A function simulator(rng) -> batch_of_data
                          or a jitted function that produces a single item.
                          We'll vmap/jit over it in a background thread.
            rng:          JAX PRNGKey to seed the simulator.
            batch_size:   Number of items per batch.
            max_queue_size: Max items the queue can hold.
            device:       "cpu" or "gpu" (passed to jax.devices()).
            device_idx:   Index of the device (0 for first GPU, etc.).
            daemon:       Whether the producer threads are daemonized.
            num_producers: Number of producer threads to use.
            queue_timeout: Timeout in seconds for queue operations (None = no timeout).
        """
        self.simulator_fn = simulator_fn
        if rng is None:
            rng = jax.random.PRNGKey(0)
        self.rng = rng
        self.batch_size = batch_size
        self.device = jax.devices(data_device)[device_idx]
        self.simulation_device = jax.devices(simulation_device)[device_idx]
        self.num_producers = max(1, num_producers)  # Ensure at least 1 producer

        # For the background threads
        self.event = threading.Event()
        self.queue = queue.Queue(maxsize=max_queue_size)
        self.queue_timeout = queue_timeout
        self.paused = threading.Event()  # For pause/resume functionality
        self.stats = {
            "batches_produced": 0,
            "batches_consumed": 0,
            "production_time": 0.0,
            "queue_wait_time": 0.0,
        }
        self.thread_exceptions = queue.Queue()

        # Create and start multiple producer threads
        self.producer_threads = []
        # Split the RNG for each thread
        thread_rngs = jax.random.split(self.rng, self.num_producers)

        for i in range(self.num_producers):
            thread_rng = thread_rngs[i]
            thread = threading.Thread(
                target=self._producer_loop,
                args=(thread_rng,),
                daemon=daemon,
                name=f"producer-{i}"
            )
            self.producer_threads.append(thread)
            thread.start()

    def _producer_loop(self, thread_rng):
        """
        The background loop that repeatedly:
          - Splits the RNG
          - Produces a batch on the device
          - Transfers it back to CPU (numpy array)
          - Puts it into the CPU queue

        Args:
            thread_rng: The initial RNG key for this thread
        """
        try:
            # We'll jit+vmap the user simulator to produce batch_size items at once
            @partial(jax.jit, device=self.simulation_device)
            def batch_simulator(rng_key):
                rngs = jax.random.split(rng_key, self.batch_size)
                return jax.vmap(self.simulator_fn)(rngs)

            key = jax.device_put(thread_rng, self.simulation_device)
            while not self.event.is_set():
                # Wait if production is paused
                if self.paused.is_set():
                    time.sleep(0.1)
                    continue

                # Generate a batch
                start_time = time.time()
                key, rng_sub = jax.random.split(key)
                data = batch_simulator(rng_sub)

                # Convert jax.Array -> NumPy (CPU) for queuing
                # so we don't hold a big device buffer in the queue
                data_cpu = np.array(data, copy=False)

                production_time = time.time() - start_time
                with threading.Lock():
                    self.stats["production_time"] += production_time

                # Blocks if queue is full
                queue_start = time.time()
                try:
                    self.queue.put(data_cpu, timeout=self.queue_timeout)
                    with threading.Lock():
                        self.stats["batches_produced"] += 1
                        self.stats["queue_wait_time"] += time.time() - queue_start
                except queue.Full:
                    # Just continue trying if we hit a timeout
                    continue
        except Exception as e:
            # Capture exceptions from threads
            self.thread_exceptions.put((threading.current_thread().name, e))
            raise  # Re-raise to see in thread

    def _cpu_data_stream(self):
        """
        A generator that yields batches from the CPU queue.
        Ends if the event is set or a thread exception occurred.
        """
        while not self.event.is_set():
            # Check for thread exceptions
            if not self.thread_exceptions.empty():
                thread_name, exception = self.thread_exceptions.get()
                raise RuntimeError(f"Exception in producer thread {thread_name}: {exception}")

            # Blocks if queue is empty
            try:
                batch_cpu = self.queue.get(timeout=self.queue_timeout)
                with threading.Lock():
                    self.stats["batches_consumed"] += 1
                yield batch_cpu
            except queue.Empty:
                # If we hit a timeout, check if we should exit
                if self.event.is_set():
                    break
                continue

    def _prefetch_data_stream(self, cpu_stream):
        """
        Generator that:
          - Pulls CPU batches from 'cpu_stream'
          - Asynchronously copies them to `self.device` using `_transfer_to_device()`
          - Yields them (already on the device)
          - Prefetches the next batch in parallel
        """
        stream_iter = iter(cpu_stream)

        # Try to grab first batch
        try:
            next_batch_on_gpu = jax.device_put(next(stream_iter), self.device)
        except StopIteration:
            return  # If no data, we're done

        while True:
            current_batch = next_batch_on_gpu  # The batch to yield
            # Prefetch the next batch
            try:
                cpu_batch = next(stream_iter)
                next_batch_on_gpu = jax.device_put(cpu_batch, self.device)
            except StopIteration:
                next_batch_on_gpu = None

            yield current_batch

            if next_batch_on_gpu is None:
                break

    def __iter__(self):
        """
        Returns a generator that yields GPU batches, prefetching them in parallel.
        """
        cpu_stream = self._cpu_data_stream()
        # `yield from` the prefetch generator
        yield from self._prefetch_data_stream(cpu_stream)

    def close(self):
        """
        Signal all producer threads to exit and wait for them to join.
        """
        self.event.set()
        for thread in self.producer_threads:
            if thread.is_alive():
                thread.join()

    def __del__(self):
        """
        Make sure the background threads are stopped if the loader is garbage-collected.
        """
        self.close()

    def queue_size(self):
        """Return the current size of the queue."""
        return self.queue.qsize()

    def queue_empty(self):
        """Check if the queue is empty."""
        return self.queue.empty()

    def queue_full(self):
        """Check if the queue is full."""
        return self.queue.full()

    def get_stats(self):
        """Return statistics about the dataloader operation."""
        return {k: v for k, v in self.stats.items()}

    def pause(self):
        """Temporarily pause data production."""
        self.paused.set()

    def resume(self):
        """Resume data production after pausing."""
        self.paused.clear()

    def __enter__(self):
        """Support for context manager protocol."""
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        """Clean up resources when exiting a context manager block."""
        self.close()
        return False  # Don't suppress exceptions


In [25]:
loader = StreaDataLoader(
    simulator_fn=example_simulator,
    rng=None,
    batch_size=5048,
    max_queue_size=10000,
    simulation_device="cpu",
    data_device="gpu",
    ring_size=2,
    use_inplace_update=False,
    daemon=True
)

NameError: name 'StreaDataLoader' is not defined

In [26]:
loader.queue.qsize()

0

In [27]:
%%timeit
next(iter(loader))


RuntimeError: Exception in producer thread producer-2: name 'example_simulator' is not defined

In [28]:
next(iter(loader)).device

RuntimeError: Exception in producer thread producer-3: name 'example_simulator' is not defined

In [29]:
def example_simulator2(key):
    return (example_simulator(key),)

In [6]:
from dmri.train.dataloader import StreamDataLoader

loader = StreamDataLoader(
    simulator_fn=example_simulator2,
    batch_size=32,
    max_queue_size=10000,
    simulation_device="cpu",
    data_device="gpu",
    use_inplace_updates=False,
    num_producers=4,
    daemon=True
)

NameError: name 'example_simulator2' is not defined

In [7]:
loader.queue.qsize()

NameError: name 'loader' is not defined

In [8]:
%%timeit
next(iter(loader))


NameError: name 'loader' is not defined

In [9]:
next(iter(loader))

NameError: name 'loader' is not defined

In [1]:
from dmri.simulators.acquisition_scheme import random_clinical_acquisition
import jax

In [2]:
jax.jit(random_clinical_acquisition)(jax.random.PRNGKey(0))

acquisition_scheme(bvals=Array([1000., 1000.,    0., 1000., 1000., 1000.,    0., 1000., 1000.,
          0., 1000., 1000., 1000., 1000.,    0., 1000., 1000., 1000.,
       1000., 1000.,    0., 1000., 1000., 1000., 1000., 1000., 1000.,
       1000., 1000., 1000., 1000., 1000., 1000.,    0.,    0.],      dtype=float32), bvecs=Array([[-0.58086044,  0.34696656, -0.7363526 ],
       [-0.07982834, -0.88641274, -0.45596057],
       [ 0.41572645,  0.8828008 ,  0.21871015],
       [-0.17049362, -0.43959284, -0.88186747],
       [ 0.58466446, -0.34851304, -0.73260224],
       [ 0.7491653 ,  0.64201397,  0.16300148],
       [ 0.12928574,  0.2528005 , -0.95884156],
       [ 0.00956197, -0.67679363,  0.7361108 ],
       [ 0.54044044, -0.05011935, -0.8398883 ],
       [-0.9110033 ,  0.0988547 , -0.40037587],
       [-0.08012941,  0.7043554 , -0.70531034],
       [-0.6081518 , -0.6734677 , -0.42022923],
       [ 0.8896543 ,  0.31788227, -0.3278203 ],
       [ 0.36490956, -0.3490052 , -0.8631549 ],
  